# Фінальний проєкт: датасет `infectious_cases` (PostgreSQL)

Notebook виконує **ті самі SQL-блоки**, що й файл `final_project.sql`: він генерується з цього файлу, тож код у них ідентичний. Для кожного запиту в `screenshots/` зберігається PNG з текстом SQL і результатом.

| Завдання | Що зроблено |
|---|---|
| 1 | схема `pandemic`, `SET search_path`, імпорт CSV у `pandemic.infectious_cases`, огляд даних |
| 2 | 3НФ: `entities` (унікальні Entity + Code) і `disease_cases` (показники з `entity_id`) + перевірка lossless-join |
| 3 | AVG / MIN / MAX / SUM для `Number_rabies` за кожною Entity, топ-10 за середнім |
| 4 | `MAKE_DATE`, `CURRENT_DATE`, різниця в роках через `AGE` |
| 5 | `fn_years_since(year)` і `fn_avg_cases_per_period(cases, divisor)` + використання й перевірка |

## 0. Середовище

In [1]:
# Install dependencies. PyPI ships pgserver wheels only for CPython 3.9-3.12, but its C extension
# is built against the stable ABI (abi3), so on Python 3.13+ we download the cp312 wheel and
# retag it as cp312-abi3, which pip accepts for any CPython >= 3.12.
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(PIP + ["psycopg2-binary", "sqlalchemy", "pandas", "matplotlib", "gdown"], check=True)

if subprocess.run(PIP + ["pgserver"], capture_output=True).returncode != 0:
    wheel_dir = Path("/tmp/pgserver_wheel")
    subprocess.run(
        [sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
         "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
         "--platform", "manylinux2014_x86_64", "-d", str(wheel_dir)],
        check=True,
    )
    src = next(wheel_dir.glob("pgserver-0.1.4-cp312-cp312-*.whl"))
    dst = src.with_name(src.name.replace("cp312-cp312", "cp312-abi3"))
    src.rename(dst)
    subprocess.run(PIP + [str(dst)], check=True)

print(f"Python {sys.version.split()[0]}: залежності встановлено")

Python 3.13.18: залежності встановлено


In [2]:
import hashlib
import re
import textwrap

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

matplotlib.use("Agg")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

pg = pgserver.get_server("/tmp/fp_pg", cleanup_mode="stop")
# psycopg2 is pinned explicitly (SQLAlchemy 2.1 defaults to psycopg 3). The search_path option makes
# every pooled connection resolve unqualified names in schema "pandemic", the same effect that
# "SET search_path TO pandemic" has in a single psql session
engine = create_engine(
    pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1),
    connect_args={"options": "-c search_path=pandemic,public"},
    future=True,
)


def q(sql: str) -> pd.DataFrame:
    # Run a SELECT and return the result; coerce_float=False keeps NUMERIC as exact Decimal
    # (e.g. 19182.80 instead of the float 19182.8)
    with engine.begin() as conn:
        return pd.read_sql(text(sql), conn, coerce_float=False)


def run_sql(sql: str) -> None:
    # Run a multi-statement script on a raw psycopg2 cursor (no parameter parsing of "%")
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur:
            cur.execute(sql)
        raw_conn.commit()
    finally:
        raw_conn.close()


with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:467: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
SCREENSHOT_DIR = Path("screenshots")
SCREENSHOT_DIR.mkdir(exist_ok=True)
FONT = "DejaVu Sans Mono"


def _render(name: str, title: str, sql: str, df: pd.DataFrame | None, max_rows: int = 12) -> Path:
    # Render the title, SQL text and (optionally) the result table into one PNG "screenshot"
    sql_lines = []
    for line in sql.splitlines():
        sql_lines.extend(textwrap.wrap(line, 110, subsequent_indent="    ") or [""])
    shown = None
    if df is not None:
        shown = df.head(max_rows).copy()
        for col in shown.columns:
            shown[col] = shown[col].map(lambda v: "NULL" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
    sql_h = 0.55 + 0.165 * len(sql_lines)
    tbl_h = 0.0 if shown is None else 0.45 + 0.27 * (len(shown) + 1)
    total_h = sql_h + tbl_h
    fig = plt.figure(figsize=(14, total_h))
    ax_sql = fig.add_axes([0, tbl_h / total_h, 1, sql_h / total_h])
    ax_sql.axis("off")
    ax_sql.text(0.005, 1.0, title, fontsize=13, fontweight="bold", family=FONT, va="top",
                transform=ax_sql.transAxes, parse_math=False)
    # parse_math=False: "$$" in function bodies must not be treated as a LaTeX formula
    ax_sql.text(0.005, 1.0 - 0.42 / sql_h, "\n".join(sql_lines), fontsize=8.5, family=FONT, va="top",
                color="#1f3b70", transform=ax_sql.transAxes, parse_math=False,
                bbox=dict(facecolor="#f3f5f9", edgecolor="#c9d1e0", boxstyle="round,pad=0.4"))
    if shown is not None:
        ax_tbl = fig.add_axes([0.005, 0, 0.99, tbl_h / total_h])
        ax_tbl.axis("off")
        tbl = ax_tbl.table(cellText=shown.values, colLabels=list(shown.columns), loc="upper left", cellLoc="left")
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8.5)
        tbl.auto_set_column_width(list(range(len(shown.columns))))
        tbl.scale(1, 1.3)
        for (row, _), cell in tbl.get_celld().items():
            cell.set_text_props(family=FONT, fontweight="bold" if row == 0 else "normal")
            if row == 0:
                cell.set_facecolor("#dde5f3")
        if len(df) > max_rows:
            ax_tbl.text(0, 0, f"... показано {max_rows} з {len(df)} рядків", fontsize=8, family=FONT,
                        transform=ax_tbl.transAxes, parse_math=False)
    path = SCREENSHOT_DIR / f"{name}.png"
    fig.savefig(path, dpi=110, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def save_screenshot(name: str, title: str, sql: str, df: pd.DataFrame) -> None:
    # Screenshot of a query together with its result
    _render(name, title, sql, df)


def save_sql_screenshot(name: str, title: str, sql: str) -> None:
    # Screenshot of a DDL/DML statement (no result set)
    _render(name, title, sql, None)

## 1. Завантаження даних

### 1.1. Файл `infectious_cases.csv`

Якщо файла немає в `data/`, він завантажується з курсового Google Drive через `gdown`. MD5 дозволяє переконатися, що використано той самий файл.

In [4]:
DATA_PATH = Path("data/infectious_cases.csv")
DRIVE_FILE_ID = "1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl"

if not DATA_PATH.exists():
    import gdown

    DATA_PATH.parent.mkdir(exist_ok=True)
    gdown.download(id=DRIVE_FILE_ID, output=str(DATA_PATH), quiet=False)

raw_bytes = DATA_PATH.read_bytes()
print(f"{DATA_PATH}: {len(raw_bytes) / 1e6:.2f} MB, md5={hashlib.md5(raw_bytes).hexdigest()}")
print(raw_bytes.decode("utf-8").splitlines()[0])

Downloading...
From: https://drive.google.com/uc?id=1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl
To: /content/data/infectious_cases.csv
100%|██████████| 541k/541k [00:00<00:00, 39.6MB/s]

data/infectious_cases.csv: 0.54 MB, md5=ad6211030e4b9058d61509c2767d5317
Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases


### 1.2. Схема `pandemic` і таблиця для імпорту

In [5]:
SQL = r"""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;

DROP FUNCTION IF EXISTS fn_years_since(INTEGER);
DROP FUNCTION IF EXISTS fn_avg_cases_per_period(NUMERIC, INTEGER);
DROP TABLE IF EXISTS disease_cases CASCADE;
DROP TABLE IF EXISTS entities CASCADE;
DROP TABLE IF EXISTS infectious_cases CASCADE;

-- Raw table mirrors the CSV 1:1. Every column is TEXT so that the import never fails
-- and the original values (including empty strings) are preserved as they are in the file.
-- Column names are the CSV header in lower case (PostgreSQL folds unquoted identifiers).
CREATE TABLE infectious_cases (
    entity               TEXT,
    code                 TEXT,
    year                 TEXT,
    number_yaws          TEXT,
    polio_cases          TEXT,
    cases_guinea_worm    TEXT,
    number_rabies        TEXT,
    number_malaria       TEXT,
    number_hiv           TEXT,
    number_tuberculosis  TEXT,
    number_smallpox      TEXT,
    number_cholera_cases TEXT
);
"""
run_sql(SQL)
save_sql_screenshot("01_create_schema", "Завдання 1. CREATE SCHEMA pandemic + SET search_path + CREATE TABLE", SQL)
print("OK")

OK


### 1.3. Імпорт CSV

Спосіб той самий, що в попередніх ДЗ: `COPY ... FROM STDIN` у staging-таблицю з колонками `TEXT`. Опції збігаються з командою `\copy` у `final_project.sql`. `FORCE_NOT_NULL` зберігає порожні поля як `''`, тобто так, як вони є у файлі. Перетворення `''` → `NULL` відбувається свідомо, у нормалізації (завдання 2).

In [6]:
COPY_SQL = """
COPY pandemic.infectious_cases FROM STDIN WITH (
    FORMAT csv,
    HEADER true,
    FORCE_NOT_NULL (entity, code, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
                    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases)
)
"""
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur, open(DATA_PATH, encoding="utf-8", newline="") as f:
        cur.copy_expert(COPY_SQL, f)
    raw_conn.commit()
finally:
    raw_conn.close()

save_sql_screenshot("02_import_copy", "Завдання 1. Імпорт CSV через COPY FROM STDIN (у psql: \\copy)", COPY_SQL.strip())
q("SELECT COUNT(*) AS imported_rows FROM pandemic.infectious_cases")

,imported_rows
0,10521


### 1.4. Огляд даних

In [7]:
SQL = r"""
SELECT *
FROM infectious_cases
LIMIT 10;
"""
res = q(SQL)
save_screenshot("03_preview_raw", "Завдання 1. Перегляд даних infectious_cases", SQL, res)
res

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,
5,Afghanistan,AFG,1985,,13867,0,,,,,,
6,Afghanistan,AFG,1986,,12901,0,,,,,,
7,Afghanistan,AFG,1987,,4396,0,,,,,,
8,Afghanistan,AFG,1988,,2149,0,,,,,,
9,Afghanistan,AFG,1989,,385,0,,,,,,


In [8]:
SQL = r"""
-- Structure overview: row count, distinct entities, year range and empty values per column
SELECT
    COUNT(*)                                           AS n_rows,
    COUNT(DISTINCT entity)                             AS n_entities,
    COUNT(DISTINCT NULLIF(code, ''))                   AS n_codes,
    MIN(year::INTEGER)                                 AS min_year,
    MAX(year::INTEGER)                                 AS max_year,
    COUNT(*) FILTER (WHERE code = '')                  AS empty_code,
    COUNT(*) FILTER (WHERE number_rabies = '')         AS empty_rabies,
    COUNT(*) FILTER (WHERE number_malaria = '')        AS empty_malaria,
    COUNT(*) FILTER (WHERE polio_cases = '')           AS empty_polio,
    COUNT(*) FILTER (WHERE number_yaws = '')           AS empty_yaws
FROM infectious_cases;
"""
res = q(SQL)
save_screenshot("04_profile_raw", "Завдання 1. Структура і пропуски", SQL, res)
res

,n_rows,n_entities,n_codes,min_year,max_year,empty_code,empty_rabies,empty_malaria,empty_polio,empty_yaws
0,10521,245,211,1920,2023,1128,3741,3741,2911,10447


**Висновок.**
- **Імпорт:** у `pandemic.infectious_cases` завантажено **10 521** рядок і 12 колонок — 245 сутностей за роки 1920–2023.
- **Пропуски** закодовано порожніми рядками: `Code` порожній у 1 128 рядках (34 агрегати на кшталт Africa, регіонів WHO і груп WB). `Number_rabies`, `Number_malaria`, `Number_hiv` і `Number_tuberculosis` порожні в одних і тих самих 3 741 рядку. Для `Number_yaws` заповнено лише 74 рядки з 10 521.
- **Змішані сутності:** у таблиці країни (ISO-3 коди) і агрегати (World — `OWID_WRL`, регіони, групи доходів). Це важливо для інтерпретації завдання 3.

## 2. Нормалізація до 3НФ

**Ключ вихідної таблиці** — `(Entity, Year)`. **Функціональні залежності:**
- `Entity → Code`: у кожної сутності рівно один код (перевірка нижче);
- `(Entity, Year) → Number_*`: показники конкретного року.

`Code` залежить лише від **частини** ключа (`Entity`), тобто це часткова залежність, яка порушує 2НФ, а отже й 3НФ. На практиці пара `(Entity, Code)` повторюється в кожному з років сутності. Декомпозиція:
- **`entities`** (`entity_id` PK, `entity_name` UNIQUE, `entity_code` UNIQUE) — унікальні комбінації Entity + Code;
- **`disease_cases`** (`entity_id` FK, `year`, показники; PK `(entity_id, year)`) — показники, де замість Entity і Code стоїть `entity_id`.

Обидві таблиці в 3НФ: в `entities` атрибути залежать лише від `entity_id`. У `disease_cases` кожен показник залежить від повного ключа `(entity_id, year)`, і між неключовими атрибутами немає залежностей.

Показники залишено в «широкому» форматі (колонка на хворобу), а не розгорнуто в EAV `(entity_id, year, disease, value)`. Кожна хвороба — окремий атрибут факту «сутність-рік», 3НФ цього не вимагає, а широкий формат зберігає типи, `CHECK` і простоту запитів.

In [9]:
SQL = r"""
CREATE TABLE entities (
    entity_id    INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity_name  TEXT NOT NULL UNIQUE,
    -- NULL for aggregates (World regions, income groups) that have no code in the source;
    -- UNIQUE still allows several NULLs
    entity_code  TEXT UNIQUE
        CHECK (entity_code ~ '^([A-Z]{3}|OWID_[A-Z]{3})$')
);

CREATE TABLE disease_cases (
    entity_id            INTEGER  NOT NULL REFERENCES entities (entity_id),
    year                 SMALLINT NOT NULL CHECK (year BETWEEN 1900 AND 2100),
    number_yaws          NUMERIC CHECK (number_yaws          >= 0),
    polio_cases          NUMERIC CHECK (polio_cases          >= 0),
    cases_guinea_worm    NUMERIC CHECK (cases_guinea_worm    >= 0),
    number_rabies        NUMERIC CHECK (number_rabies        >= 0),
    number_malaria       NUMERIC CHECK (number_malaria       >= 0),
    number_hiv           NUMERIC CHECK (number_hiv           >= 0),
    number_tuberculosis  NUMERIC CHECK (number_tuberculosis  >= 0),
    number_smallpox      NUMERIC CHECK (number_smallpox      >= 0),
    number_cholera_cases NUMERIC CHECK (number_cholera_cases >= 0),
    PRIMARY KEY (entity_id, year)
);
"""
run_sql(SQL)
save_sql_screenshot("05_create_normalized", "Завдання 2. DDL нормалізованих таблиць", SQL)
print("OK")

OK


Перевірка функціональної залежності `Entity → Code` до декомпозиції:

In [10]:
SQL = r"""
-- Verify the functional dependency entity -> code before decomposing:
-- both counts must be 0, otherwise entities(entity_name UNIQUE) would lose information
SELECT
    (SELECT COUNT(*) FROM (
        SELECT entity FROM infectious_cases GROUP BY entity HAVING COUNT(DISTINCT code) > 1
    ) AS x) AS entities_with_several_codes,
    (SELECT COUNT(*) FROM (
        SELECT code FROM infectious_cases WHERE code <> '' GROUP BY code HAVING COUNT(DISTINCT entity) > 1
    ) AS y) AS codes_with_several_entities,
    (SELECT COUNT(*) FROM (
        SELECT entity, year FROM infectious_cases GROUP BY entity, year HAVING COUNT(*) > 1
    ) AS z) AS duplicate_entity_year;
"""
res = q(SQL)
save_screenshot("06_fd_check", "Завдання 2. Перевірка Entity -> Code", SQL, res)
res

,entities_with_several_codes,codes_with_several_entities,duplicate_entity_year
0,0,0,0


In [11]:
assert (res.iloc[0] == 0).all(), "залежність Entity -> Code порушена: декомпозиція втратила б дані"
print("Entity -> Code: функціональна залежність виконується")

Entity -> Code: функціональна залежність виконується


In [12]:
SQL = r"""
-- Empty strings become SQL NULL ("no data"), and numeric text is cast to NUMERIC
INSERT INTO entities (entity_name, entity_code)
SELECT DISTINCT
    entity,
    NULLIF(code, '')
FROM infectious_cases
ORDER BY entity;

INSERT INTO disease_cases (
    entity_id, year,
    number_yaws, polio_cases, cases_guinea_worm, number_rabies, number_malaria,
    number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
)
SELECT
    e.entity_id,
    ic.year::SMALLINT,
    NULLIF(ic.number_yaws, '')::NUMERIC,
    NULLIF(ic.polio_cases, '')::NUMERIC,
    NULLIF(ic.cases_guinea_worm, '')::NUMERIC,
    NULLIF(ic.number_rabies, '')::NUMERIC,
    NULLIF(ic.number_malaria, '')::NUMERIC,
    NULLIF(ic.number_hiv, '')::NUMERIC,
    NULLIF(ic.number_tuberculosis, '')::NUMERIC,
    NULLIF(ic.number_smallpox, '')::NUMERIC,
    NULLIF(ic.number_cholera_cases, '')::NUMERIC
FROM infectious_cases AS ic
JOIN entities AS e
    ON e.entity_name = ic.entity;
"""
run_sql(SQL)
save_sql_screenshot("07_load_normalized", "Завдання 2. Наповнення нормалізованих таблиць", SQL)
print("OK")

OK


In [13]:
SQL = r"""
SELECT 'infectious_cases (raw)' AS table_name, COUNT(*) AS n_rows FROM infectious_cases
UNION ALL
SELECT 'entities', COUNT(*) FROM entities
UNION ALL
SELECT 'disease_cases', COUNT(*) FROM disease_cases;
"""
res = q(SQL)
save_screenshot("08_counts", "Завдання 2. Кількість рядків", SQL, res)
res

,table_name,n_rows
0,infectious_cases (raw),10521
1,entities,245
2,disease_cases,10521


**Lossless-join:** з'єднання двох нормалізованих таблиць має точно відтворити вихідну таблицю (0 рядків розбіжностей в обидва боки).

In [14]:
SQL = r"""
-- Lossless-join check: joining the two normalized tables back must reproduce the raw table
-- exactly, in both directions (0 rows on each side)
WITH rebuilt AS (
    SELECT
        e.entity_name AS entity,
        COALESCE(e.entity_code, '') AS code,
        dc.year::TEXT AS year,
        dc.number_rabies,
        dc.number_malaria,
        dc.polio_cases
    FROM disease_cases AS dc
    JOIN entities AS e USING (entity_id)
),
original AS (
    SELECT
        entity,
        code,
        year,
        NULLIF(number_rabies, '')::NUMERIC  AS number_rabies,
        NULLIF(number_malaria, '')::NUMERIC AS number_malaria,
        NULLIF(polio_cases, '')::NUMERIC    AS polio_cases
    FROM infectious_cases
)
SELECT
    (SELECT COUNT(*) FROM (SELECT * FROM original EXCEPT SELECT * FROM rebuilt) AS a) AS only_in_raw,
    (SELECT COUNT(*) FROM (SELECT * FROM rebuilt EXCEPT SELECT * FROM original) AS b) AS only_in_normalized;
"""
res = q(SQL)
save_screenshot("09_lossless", "Завдання 2. Перевірка lossless-join", SQL, res)
res

,only_in_raw,only_in_normalized
0,0,0


In [15]:
assert res.only_in_raw[0] == 0 and res.only_in_normalized[0] == 0
print("Нормалізовані таблиці відтворюють вихідну без втрат")

Нормалізовані таблиці відтворюють вихідну без втрат


In [16]:
SQL = r"""
SELECT
    dc.entity_id,
    e.entity_name,
    e.entity_code,
    dc.year,
    dc.number_rabies,
    dc.number_malaria,
    dc.polio_cases
FROM disease_cases AS dc
JOIN entities AS e USING (entity_id)
ORDER BY e.entity_name, dc.year
LIMIT 10;
"""
res = q(SQL)
save_screenshot("10_preview_normalized", "Завдання 2. Нормалізовані дані (JOIN entities)", SQL, res)
res

,entity_id,entity_name,entity_code,year,number_rabies,number_malaria,polio_cases
0,1,Afghanistan,AFG,1960,None,None,None
1,1,Afghanistan,AFG,1965,None,None,None
2,1,Afghanistan,AFG,1980,None,None,6160
3,1,Afghanistan,AFG,1981,None,None,5859
4,1,Afghanistan,AFG,1982,None,None,9730
5,1,Afghanistan,AFG,1983,None,None,13937
6,1,Afghanistan,AFG,1984,None,None,3864
7,1,Afghanistan,AFG,1985,None,None,13867
8,1,Afghanistan,AFG,1986,None,None,12901
9,1,Afghanistan,AFG,1987,None,None,4396


**Висновок.**
- **Перевірка залежності:** `Entity → Code` виконується (0 / 0 / 0), тож декомпозиція коректна.
- **Результат:** `entities` має 245 рядків (по одному на сутність, 34 з `entity_code = NULL`), `disease_cases` — 10 521 рядок з PK `(entity_id, year)`.
- **Lossless-join:** з'єднання таблиць точно відтворює вихідну (0 рядків розбіжностей в обидва боки).
- **Типи:** порожні рядки свідомо перетворено на SQL `NULL` («даних немає»), а текст — на `NUMERIC`. `CHECK (... >= 0)` і `CHECK` формату коду тепер захищають дані від некоректних значень.

## 3. Аналіз `Number_rabies`

In [17]:
SQL = r"""
SELECT
    e.entity_id,
    e.entity_name,
    e.entity_code,
    ROUND(AVG(dc.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(dc.number_rabies), 2) AS min_rabies,
    ROUND(MAX(dc.number_rabies), 2) AS max_rabies,
    ROUND(SUM(dc.number_rabies), 2) AS sum_rabies,
    COUNT(dc.number_rabies)         AS n_years
FROM disease_cases AS dc
JOIN entities AS e USING (entity_id)
WHERE dc.number_rabies IS NOT NULL
GROUP BY e.entity_id, e.entity_name, e.entity_code
ORDER BY AVG(dc.number_rabies) DESC
LIMIT 10;
"""
res = q(SQL)
save_screenshot("11_rabies_top10", "Завдання 3. AVG/MIN/MAX/SUM Number_rabies, топ-10 за середнім", SQL, res)
res

,entity_id,entity_name,entity_code,avg_rabies,min_rabies,max_rabies,sum_rabies,n_years
0,241,World,OWID_WRL,20192.37,14075.51,24744.66,605771.11,30
1,120,Lower Middle Income (WB),None,15193.96,10202.53,19182.80,455818.78,30
2,202,South Asia (WB),None,11729.89,7271.28,15361.88,351896.66,30
3,204,South-East Asia Region (WHO),None,11424.33,6806.01,15641.96,342729.81,30
4,77,G20,None,10189.05,6339.08,13164.88,305671.38,30
5,97,India,IND,8599.17,5425.87,11121.14,257975.20,30
6,209,Sub-Saharan Africa (WB),None,4748.27,4356.82,5017.98,142448.04,30
7,3,African Region (WHO),None,4649.70,4283.07,4917.78,139491.07,30
8,121,Low Income (WB),None,3568.11,3014.62,3873.53,107043.16,30
9,58,East Asia & Pacific (WB),None,3427.46,2009.94,4591.01,102823.83,30


In [18]:
assert len(res) == 10 and res.avg_rabies.is_monotonic_decreasing and res.avg_rabies.notna().all()

**Чому потрібен `WHERE number_rabies IS NOT NULL`.** У 19 сутностей даних про rabies немає взагалі, і `AVG` для них — `NULL`. У PostgreSQL `ORDER BY ... DESC` за замовчуванням ставить `NULLS FIRST`, тож без фільтра «топ-10» виглядав би так:

In [19]:
SQL = r"""
-- The same aggregation without the IS NOT NULL filter: NULL averages float to the top
SELECT
    e.entity_name,
    e.entity_code,
    AVG(dc.number_rabies) AS avg_rabies
FROM disease_cases AS dc
JOIN entities AS e USING (entity_id)
GROUP BY e.entity_name, e.entity_code
ORDER BY avg_rabies DESC
LIMIT 10;
"""
res = q(SQL)
save_screenshot("12_rabies_nulls_pitfall", "Завдання 3. Пастка: без фільтра NULL опиняються першими", SQL, res)
res

,entity_name,entity_code,avg_rabies
0,Western Pacific,None,None
1,Europe,None,None
2,Reunion,REU,None
3,Hong Kong,HKG,None
4,Americas,None,None
5,Saint Helena,SHN,None
6,Macao,MAC,None
7,Oceania,None,None
8,Africa,None,None
9,Singapore,SGP,None


Топ-10 займають агрегати (World, регіони WB/WHO, G20). Додатково — лише країни (ISO-3 коди):

In [20]:
SQL = r"""
-- Additional view: the top 10 is dominated by aggregates (World, WB/WHO regions, G20);
-- restricting to ISO-3 country codes shows individual countries
SELECT
    e.entity_name,
    e.entity_code,
    ROUND(AVG(dc.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(dc.number_rabies), 2) AS min_rabies,
    ROUND(MAX(dc.number_rabies), 2) AS max_rabies,
    ROUND(SUM(dc.number_rabies), 2) AS sum_rabies
FROM disease_cases AS dc
JOIN entities AS e USING (entity_id)
WHERE dc.number_rabies IS NOT NULL
  AND e.entity_code ~ '^[A-Z]{3}$'
GROUP BY e.entity_name, e.entity_code
ORDER BY AVG(dc.number_rabies) DESC
LIMIT 10;
"""
res = q(SQL)
save_screenshot("13_rabies_countries", "Завдання 3 (додатково). Лише країни", SQL, res)
res

,entity_name,entity_code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,India,IND,8599.17,5425.87,11121.14,257975.20
1,Pakistan,PAK,1582.17,1177.14,1881.73,47465.09
2,Nigeria,NGA,1335.42,1073.95,1441.12,40062.46
3,China,CHN,1247.91,744.25,2075.59,37437.20
4,Ethiopia,ETH,1145.17,758.05,1323.40,34355.18
5,Myanmar,MMR,972.90,560.70,1449.83,29187.11
6,Bangladesh,BGD,785.71,180.40,1734.71,23571.26
7,Nepal,NPL,661.25,338.14,1041.63,19837.53
8,Philippines,PHL,618.69,358.06,898.07,18560.73
9,Indonesia,IDN,267.71,124.62,481.37,8031.44


**Висновок.**
- **Топ-10 за середнім `Number_rabies`** очолює World (≈ 20 192 на рік). Далі йдуть агрегати (Lower Middle Income, South Asia, South-East Asia Region WHO, G20), а **India** — єдина країна в топі (≈ 8 599).
- **Пастка `NULLS FIRST`.** Без `WHERE number_rabies IS NOT NULL` перші 10 рядків — сутності з `AVG = NULL` (Western Pacific, Europe, Hong Kong…), бо в PostgreSQL `DESC` за замовчуванням ставить `NULL` першими.
- **Лише країни:** India, Pakistan, Nigeria, China, Ethiopia… Індія за середнім перевищує другу країну (Пакистан) більш ніж у 5 разів.
- **Природа даних:** `n_years = 30` (1990–2019) і дробові значення вказують на модельні оцінки (типу IHME GBD), а не на зареєстровані випадки.

## 4. Різниця в роках (вбудовані функції)

In [21]:
SQL = r"""
SELECT
    e.entity_name,
    dc.year,
    MAKE_DATE(dc.year, 1, 1) AS first_jan_date,
    CURRENT_DATE AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(dc.year, 1, 1)))::INTEGER AS years_diff
FROM disease_cases AS dc
JOIN entities AS e USING (entity_id)
ORDER BY e.entity_name, dc.year
LIMIT 10;
"""
res = q(SQL)
save_screenshot("14_year_diff", "Завдання 4. MAKE_DATE, CURRENT_DATE, AGE", SQL, res)
res

,entity_name,year,first_jan_date,today,years_diff
0,Afghanistan,1960,1960-01-01,2026-11-05,66
1,Afghanistan,1965,1965-01-01,2026-11-05,61
2,Afghanistan,1980,1980-01-01,2026-11-05,46
3,Afghanistan,1981,1981-01-01,2026-11-05,45
4,Afghanistan,1982,1982-01-01,2026-11-05,44
5,Afghanistan,1983,1983-01-01,2026-11-05,43
6,Afghanistan,1984,1984-01-01,2026-11-05,42
7,Afghanistan,1985,1985-01-01,2026-11-05,41
8,Afghanistan,1986,1986-01-01,2026-11-05,40
9,Afghanistan,1987,1987-01-01,2026-11-05,39


**Висновок.** `MAKE_DATE(year, 1, 1)` будує 1 січня року, `CURRENT_DATE` повертає дату запуску, а `EXTRACT(YEAR FROM AGE(...))` дає кількість **повних** років між ними. Оскільки перша дата — завжди 1 січня, результат дорівнює `поточний рік − year` (наприклад, 1980 → 46 при запуску в 2026). Значення залежить від дати виконання, тож після повторного запуску в іншому році воно зміниться.

## 5. Власні функції

In [22]:
SQL = r"""
-- Years between January 1 of p_year and today.
-- STABLE (not IMMUTABLE): the result depends on CURRENT_DATE, i.e. on the day it is called.
-- STRICT: NULL input -> NULL result without evaluating the body.
CREATE OR REPLACE FUNCTION fn_years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
STABLE
STRICT
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
$$;

-- Average number of cases per period: yearly cases divided by an arbitrary divisor
-- (12 = month, 4 = quarter, 2 = half-year).
-- IMMUTABLE: depends only on its arguments. A divisor <= 0 is rejected explicitly
-- instead of failing with a generic "division by zero" or returning a negative average.
CREATE OR REPLACE FUNCTION fn_avg_cases_per_period(p_cases_per_year NUMERIC, p_divisor INTEGER)
RETURNS NUMERIC
LANGUAGE plpgsql
IMMUTABLE
STRICT
AS $$
BEGIN
    IF p_divisor <= 0 THEN
        RAISE EXCEPTION 'p_divisor must be a positive integer, got %', p_divisor;
    END IF;
    RETURN p_cases_per_year / p_divisor;
END;
$$;
"""
run_sql(SQL)
save_sql_screenshot("15_create_functions", "Завдання 5. CREATE FUNCTION", SQL)
print("OK")

OK


### 5.1. `fn_years_since(year)`

In [23]:
SQL = r"""
SELECT
    e.entity_name,
    dc.year,
    fn_years_since(dc.year) AS years_since
FROM disease_cases AS dc
JOIN entities AS e USING (entity_id)
ORDER BY e.entity_name, dc.year
LIMIT 10;
"""
res = q(SQL)
save_screenshot("16_fn_years_since", "Завдання 5. Використання fn_years_since", SQL, res)
res

,entity_name,year,years_since
0,Afghanistan,1960,66
1,Afghanistan,1965,61
2,Afghanistan,1980,46
3,Afghanistan,1981,45
4,Afghanistan,1982,44
5,Afghanistan,1983,43
6,Afghanistan,1984,42
7,Afghanistan,1985,41
8,Afghanistan,1986,40
9,Afghanistan,1987,39


In [24]:
SQL = r"""
-- The function must agree with the built-in expression from Task 4 for every row
SELECT
    COUNT(*) AS rows_checked,
    COUNT(*) FILTER (
        WHERE fn_years_since(year)
              IS DISTINCT FROM EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER
    ) AS mismatches,
    fn_years_since(1996) AS example_1996
FROM disease_cases;
"""
res = q(SQL)
save_screenshot("17_fn_years_since_check", "Завдання 5. Перевірка fn_years_since", SQL, res)
res

,rows_checked,mismatches,example_1996
0,10521,0,30


In [25]:
assert res.mismatches[0] == 0
print(f"fn_years_since збігається з вбудованим виразом у {res.rows_checked[0]} рядках; 1996 -> {res.example_1996[0]}")

fn_years_since збігається з вбудованим виразом у 10521 рядках; 1996 -> 30


### 5.2. `fn_avg_cases_per_period(cases, divisor)`

Використовується лише для записів, де кількість захворювань **числова**. На сирій TEXT-таблиці фільтр — регулярний вираз `'^[0-9]+(\.[0-9]+)?$'`. Він відкидає порожні рядки (та будь-який інший нечисловий текст) **до** приведення типу, тож `::NUMERIC` ніколи не падає.

In [26]:
SQL = r"""
-- Only records whose number of cases is numeric: on the raw TEXT table the regex filter
-- drops empty strings (and would drop any other non-numeric text) before the cast
SELECT
    entity,
    year,
    number_rabies::NUMERIC AS rabies_per_year,
    ROUND(fn_avg_cases_per_period(number_rabies::NUMERIC, 12), 4) AS avg_per_month,
    ROUND(fn_avg_cases_per_period(number_rabies::NUMERIC, 4), 4)  AS avg_per_quarter,
    ROUND(fn_avg_cases_per_period(number_rabies::NUMERIC, 2), 4)  AS avg_per_half_year
FROM infectious_cases
WHERE number_rabies ~ '^[0-9]+(\.[0-9]+)?$'
ORDER BY entity, year
LIMIT 10;
"""
res = q(SQL)
save_screenshot("18_fn_avg_cases", "Завдання 5. fn_avg_cases_per_period: 12 / 4 / 2", SQL, res)
res

,entity,year,rabies_per_year,avg_per_month,avg_per_quarter,avg_per_half_year
0,Afghanistan,1990,2.145772,0.1788,0.5364,1.0729
1,Afghanistan,1991,2.3209748,0.1934,0.5802,1.1605
2,Afghanistan,1992,2.6180408,0.2182,0.6545,1.3090
3,Afghanistan,1993,2.8531172,0.2378,0.7133,1.4266
4,Afghanistan,1994,2.9899871,0.2492,0.7475,1.4950
5,Afghanistan,1995,3.126361,0.2605,0.7816,1.5632
6,Afghanistan,1996,3.281437,0.2735,0.8204,1.6407
7,Afghanistan,1997,3.446122,0.2872,0.8615,1.7231
8,Afghanistan,1998,3.5570247,0.2964,0.8893,1.7785
9,Afghanistan,1999,3.607037,0.3006,0.9018,1.8035


In [27]:
SQL = r"""
-- Consistency: the numeric filter on the raw table selects exactly the non-NULL values of the
-- normalized table, and month * 12 = quarter * 4 = half-year * 2 = yearly value for every row.
-- NUMERIC division is rounded to a finite scale (2.145772 / 12 * 12 = 2.14577199999...),
-- so the round trip is compared with a tolerance instead of exact equality
SELECT
    (SELECT COUNT(*) FROM infectious_cases WHERE number_rabies ~ '^[0-9]+(\.[0-9]+)?$') AS numeric_raw_rows,
    (SELECT COUNT(*) FROM infectious_cases WHERE number_rabies = '')                    AS empty_raw_rows,
    (SELECT COUNT(number_rabies) FROM disease_cases)                                    AS not_null_normalized_rows,
    (SELECT COUNT(*) FROM disease_cases
     WHERE number_rabies IS NOT NULL
       AND GREATEST(
               ABS(fn_avg_cases_per_period(number_rabies, 12) * 12 - number_rabies),
               ABS(fn_avg_cases_per_period(number_rabies, 4)  * 4  - number_rabies),
               ABS(fn_avg_cases_per_period(number_rabies, 2)  * 2  - number_rabies)
           ) > 1e-9
    ) AS period_mismatches;
"""
res = q(SQL)
save_screenshot("19_fn_avg_cases_check", "Завдання 5. Перевірка fn_avg_cases_per_period", SQL, res)
res

,numeric_raw_rows,empty_raw_rows,not_null_normalized_rows,period_mismatches
0,6780,3741,6780,0


In [28]:
assert res.numeric_raw_rows[0] == res.not_null_normalized_rows[0]
assert res.numeric_raw_rows[0] + res.empty_raw_rows[0] == q("SELECT COUNT(*) AS n FROM infectious_cases").n[0]
assert res.period_mismatches[0] == 0

Некоректний дільник відхиляється явно, а не повертає від'ємне чи нескінченне «середнє»:

In [29]:
for divisor in (0, -3):
    try:
        q(f"SELECT fn_avg_cases_per_period(100, {divisor})")
        print(f"divisor={divisor}: помилки немає (неочікувано)")
    except Exception as exc:
        # pandas wraps the driver error; take the PL/pgSQL RAISE message from its text
        print(f"divisor={divisor}: {re.search(r'p_divisor must[^\n]*', str(exc)).group(0)}")
q("SELECT fn_avg_cases_per_period(NULL, 12) AS null_input_gives_null")

divisor=0: p_divisor must be a positive integer, got 0
divisor=-3: p_divisor must be a positive integer, got -3


,null_input_gives_null
0,None


**Висновок.**
- **`fn_years_since(year)`** (`LANGUAGE sql`, `STABLE STRICT`) збігається з вбудованим виразом із завдання 4 в усіх 10 521 рядку. Для прикладу з умови `fn_years_since(1996) = 30` при запуску в 2026 році. Функція `STABLE`, а не `IMMUTABLE`, бо залежить від `CURRENT_DATE`: позначка `IMMUTABLE` дозволила б PostgreSQL закешувати результат і повертати застарілу різницю.
- **`fn_avg_cases_per_period(cases, divisor)`** (`plpgsql`, `IMMUTABLE STRICT`) для дільників 12 / 4 / 2 дає середнє на місяць, квартал і півріччя (Afghanistan 1990: 2,1458 → 0,1788 / 0,5364 / 1,0729).
- **Лише числові записи:** regex-фільтр на сирій таблиці відбирає **6 780** рядків, що точно дорівнює кількості не-`NULL` у нормалізованій таблиці. Порожніх рядків 3 741, разом 10 521. Зворотна перевірка `місяць × 12 = квартал × 4 = півріччя × 2 = рік` проходить для всіх рядків (з допуском 1e-9 на округлення `NUMERIC`).
- **Некоректні вхідні дані:** дільник ≤ 0 відхиляється явною помилкою, а `NULL` на вході дає `NULL`.

## Згенеровані файли

- `final_project.sql` — увесь SQL проєкту; запускається через `psql -f final_project.sql`;
- `screenshots/*.png` — SQL і результат кожного кроку;
- `data/infectious_cases.csv` — вихідний датасет.

In [30]:
sorted(p.name for p in SCREENSHOT_DIR.glob("*.png"))

['01_create_schema.png',
 '02_import_copy.png',
 '03_preview_raw.png',
 '04_profile_raw.png',
 '05_create_normalized.png',
 '06_fd_check.png',
 '07_load_normalized.png',
 '08_counts.png',
 '09_lossless.png',
 '10_preview_normalized.png',
 '11_rabies_top10.png',
 '12_rabies_nulls_pitfall.png',
 '13_rabies_countries.png',
 '14_year_diff.png',
 '15_create_functions.png',
 '16_fn_years_since.png',
 '17_fn_years_since_check.png',
 '18_fn_avg_cases.png',
 '19_fn_avg_cases_check.png']